# API (OpenAlex)

In [22]:
import requests
import pandas as pd
import time
import random
from pathlib import Path
import json

QUERY = "feminism"
MAX_RESULTS = 10000
SAVE_EVERY = 50
BASE_DELAY = 1.5
JITTER = 0.7

OUT_CSV = Path("/Users/huss/data-processing/week11/Project/openalex_publications.csv")
OUT_MD = Path("/Users/huss/data-processing/week11/Project/openalex_publications_summary.md")

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
OUT_MD.parent.mkdir(parents=True, exist_ok=True)

def fetch_openalex_works(query, cursor="*"):
    url = "https://api.openalex.org/works"
    params = {
        "search": query,
        "per_page": 200,
        "cursor": cursor
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def safe_join(values, sep="; "):
    return sep.join([v for v in values if v])

def get_authors_and_order(work):
    authorships = work.get("authorships") or []
    names = []
    ordered = []
    for i, a in enumerate(authorships, 1):
        author = a.get("author") or {}
        name = author.get("display_name")
        if name:
            names.append(name)
            ordered.append(f"{i}. {name}")
    return safe_join(names), safe_join(ordered)

def get_institutions(work):
    authorships = work.get("authorships") or []
    insts = []
    for a in authorships:
        for inst in (a.get("institutions") or []):
            name = inst.get("display_name")
            if name:
                insts.append(name)
    return safe_join(sorted(set(insts)))

def get_venue(work):
    primary_location = work.get("primary_location") or {}
    source = primary_location.get("source") or {}
    venue = source.get("display_name")
    if venue:
        return venue

    best_oa_location = work.get("best_oa_location") or {}
    source = best_oa_location.get("source") or {}
    venue = source.get("display_name")
    if venue:
        return venue

    locations = work.get("locations") or []
    for loc in locations:
        source = (loc or {}).get("source") or {}
        venue = source.get("display_name")
        if venue:
            return venue

    return "N/A"

def get_abstract_text(work):
    idx = work.get("abstract_inverted_index")
    if not idx:
        return "N/A"
    words = []
    for word, positions in idx.items():
        for pos in positions:
            words.append((pos, word))
    words.sort(key=lambda x: x[0])
    return " ".join([w for _, w in words])

def get_concepts(work):
    concepts = work.get("concepts") or []
    return safe_join([c.get("display_name", "") for c in concepts])

def get_topics(work):
    topics = work.get("topics") or []
    return safe_join([t.get("display_name", "") for t in topics])

def get_fields_subfields(work):
    domains = work.get("primary_topic") or {}
    if isinstance(domains, dict):
        field = domains.get("field", {}) or {}
        subfield = domains.get("subfield", {}) or {}
        domain = domains.get("domain", {}) or {}
        return {
            "Domain": domain.get("display_name", "N/A"),
            "Field": field.get("display_name", "N/A"),
            "Subfield": subfield.get("display_name", "N/A")
        }
    return {"Domain": "N/A", "Field": "N/A", "Subfield": "N/A"}

def get_referenced_works(work):
    refs = work.get("referenced_works") or []
    return safe_join(refs)

def get_related_works(work):
    rel = work.get("related_works") or []
    return safe_join(rel)

def get_languages(work):
    langs = work.get("language")
    if isinstance(langs, list):
        return safe_join(langs)
    return langs or "N/A"

def get_publisher(work):
    primary_location = work.get("primary_location") or {}
    source = primary_location.get("source") or {}
    publisher = source.get("host_organization_name")
    if publisher:
        return publisher
    return work.get("publisher", "N/A")

def get_oa_fields(work):
    oa = work.get("open_access") or {}
    return oa.get("is_oa", False), oa.get("oa_status", "N/A")

def normalize_record(idx, work):
    authors, author_order = get_authors_and_order(work)
    institutions = get_institutions(work)
    is_oa, oa_status = get_oa_fields(work)
    tf = get_fields_subfields(work)

    return {
        "ID": idx,
        "Title": work.get("title", "N/A"),
        "Year": work.get("publication_year", "N/A"),
        "Citations": work.get("cited_by_count", 0),
        "Authors": authors[:120],
        "AuthorOrder": author_order[:250],
        "Venue": get_venue(work),
        "AbstractInvertedIndex": json.dumps(work.get("abstract_inverted_index", {}))[:1000],
        "AbstractText": get_abstract_text(work)[:1000],
        "Publisher": get_publisher(work),
        "OpenAccess": is_oa,
        "OAStatus": oa_status,
        "ReferencedWorks": get_referenced_works(work)[:1000],
        "RelatedWorks": get_related_works(work)[:1000],
        "Concepts": get_concepts(work)[:500],
        "Topics": get_topics(work)[:500],
        "Domain": tf["Domain"],
        "Field": tf["Field"],
        "Subfield": tf["Subfield"],
        "Institutions": institutions[:250],
        "Languages": get_languages(work),
        "WorkType": work.get("type", "N/A"),
        "DOI": work.get("doi", "N/A"),
        "LandingPage": (work.get("primary_location") or {}).get("landing_page_url", "N/A"),
        "PDFURL": (work.get("primary_location") or {}).get("pdf_url", "N/A")
    }

records = []
seen_ids = set()
cursor = "*"
idx = 0
total_found = None

while idx < MAX_RESULTS:
    try:
        data = fetch_openalex_works(QUERY, cursor=cursor)
        if total_found is None:
            total_found = data.get("meta", {}).get("count", 0)

        results = data.get("results", [])
        if not results:
            print("No more results.")
            break

        for work in results:
            if idx >= MAX_RESULTS:
                break

            work_id = work.get("id")
            if work_id and work_id in seen_ids:
                continue
            if work_id:
                seen_ids.add(work_id)

            idx += 1
            record = normalize_record(idx, work)
            records.append(record)

            print(f"Collected: {record['Title']}")

            if len(records) % SAVE_EVERY == 0:
                pd.DataFrame(records).to_csv(OUT_CSV, index=False)
                print(f"Saved checkpoint: {len(records)} rows")

            sleep_time = BASE_DELAY + random.uniform(-JITTER, JITTER)
            time.sleep(max(0, sleep_time))

        cursor = data.get("meta", {}).get("next_cursor")
        if not cursor:
            break

    except Exception as e:
        print(f"Skipped result due to error: {e}")
        sleep_time = BASE_DELAY + random.uniform(-JITTER, JITTER)
        time.sleep(max(0, sleep_time))

df = pd.DataFrame(records)
df.to_csv(OUT_CSV, index=False)

print(df.to_string(index=False))
print(f"Saved CSV: {OUT_CSV}")
print(f"Total records collected: {len(df)}")
print(f"Total records available for query: {total_found}")

summary = [
    "# OpenAlex Publication Summary",
    "",
    f"- Query: {QUERY}",
    f"- Total collected: {len(df)}",
    f"- Total available: {total_found}",
    f"- Unique works: {df['Title'].nunique() if not df.empty else 0}",
    f"- Average citations: {round(df['Citations'].mean(), 2) if not df.empty else 0}",
    f"- Max citations: {df['Citations'].max() if not df.empty else 0}",
    "",
    "## Sample rows",
    "",
    df.head(15).to_markdown(index=False) if not df.empty else "No rows collected."
]

OUT_MD.write_text("\n".join(summary), encoding="utf-8")
print(f"Saved markdown summary: {OUT_MD}")


Collected: Gender Trouble: Feminism and the Subversion of Identity
Collected: Situated Knowledges: The Science Question in Feminism and the Privilege of Partial Perspective


KeyboardInterrupt: 

# WebScraping (GoodReads)

In [ ]:
import requests
import pandas as pd
import time
import random
from pathlib import Path
import json

QUERY = "feminism"
SKIP_FIRST = 20000  # Skip first 10,000 results
MAX_RESULTS = 60000  # Collect 10,000 results AFTER skipping
SAVE_EVERY = 50
BASE_DELAY = 1.5
JITTER = 0.7

OUT_CSV = Path("/Users/huss/data-processing/week11/Project/openalex_publications.csv")
OUT_MD = Path("/Users/huss/data-processing/week11/Project/openalex_publications_summary.md")

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
OUT_MD.parent.mkdir(parents=True, exist_ok=True)

def fetch_openalex_works(query, cursor="*"):
    url = "https://api.openalex.org/works"
    params = {
        "search": query,
        "per_page": 200,
        "cursor": cursor
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def safe_join(values, sep="; "):
    return sep.join([v for v in values if v])

def get_authors_and_order(work):
    authorships = work.get("authorships") or []
    names = []
    ordered = []
    for i, a in enumerate(authorships, 1):
        author = a.get("author") or {}
        name = author.get("display_name")
        if name:
            names.append(name)
            ordered.append(f"{i}. {name}")
    return safe_join(names), safe_join(ordered)

def get_institutions(work):
    authorships = work.get("authorships") or []
    insts = []
    for a in authorships:
        for inst in (a.get("institutions") or []):
            name = inst.get("display_name")
            if name:
                insts.append(name)
    return safe_join(sorted(set(insts)))

def get_venue(work):
    primary_location = work.get("primary_location") or {}
    source = primary_location.get("source") or {}
    venue = source.get("display_name")
    if venue:
        return venue

    best_oa_location = work.get("best_oa_location") or {}
    source = best_oa_location.get("source") or {}
    venue = source.get("display_name")
    if venue:
        return venue

    locations = work.get("locations") or []
    for loc in locations:
        source = (loc or {}).get("source") or {}
        venue = source.get("display_name")
        if venue:
            return venue

    return "N/A"

def get_abstract_text(work):
    idx = work.get("abstract_inverted_index")
    if not idx:
        return "N/A"
    words = []
    for word, positions in idx.items():
        for pos in positions:
            words.append((pos, word))
    words.sort(key=lambda x: x[0])
    return " ".join([w for _, w in words])

def get_concepts(work):
    concepts = work.get("concepts") or []
    return safe_join([c.get("display_name", "") for c in concepts])

def get_topics(work):
    topics = work.get("topics") or []
    return safe_join([t.get("display_name", "") for t in topics])

def get_fields_subfields(work):
    domains = work.get("primary_topic") or {}
    if isinstance(domains, dict):
        field = domains.get("field", {}) or {}
        subfield = domains.get("subfield", {}) or {}
        domain = domains.get("domain", {}) or {}
        return {
            "Domain": domain.get("display_name", "N/A"),
            "Field": field.get("display_name", "N/A"),
            "Subfield": subfield.get("display_name", "N/A")
        }
    return {"Domain": "N/A", "Field": "N/A", "Subfield": "N/A"}

def get_referenced_works(work):
    refs = work.get("referenced_works") or []
    return safe_join(refs)

def get_related_works(work):
    rel = work.get("related_works") or []
    return safe_join(rel)

def get_languages(work):
    langs = work.get("language")
    if isinstance(langs, list):
        return safe_join(langs)
    return langs or "N/A"

def get_publisher(work):
    primary_location = work.get("primary_location") or {}
    source = primary_location.get("source") or {}
    publisher = source.get("host_organization_name")
    if publisher:
        return publisher
    return work.get("publisher", "N/A")

def get_oa_fields(work):
    oa = work.get("open_access") or {}
    return oa.get("is_oa", False), oa.get("oa_status", "N/A")

def normalize_record(idx, work):
    authors, author_order = get_authors_and_order(work)
    institutions = get_institutions(work)
    is_oa, oa_status = get_oa_fields(work)
    tf = get_fields_subfields(work)

    return {
        "ID": idx,
        "Title": work.get("title", "N/A"),
        "Year": work.get("publication_year", "N/A"),
        "Citations": work.get("cited_by_count", 0),
        "Authors": authors[:120],
        "AuthorOrder": author_order[:250],
        "Venue": get_venue(work),
        "AbstractInvertedIndex": json.dumps(work.get("abstract_inverted_index", {}))[:1000],
        "AbstractText": get_abstract_text(work)[:1000],
        "Publisher": get_publisher(work),
        "OpenAccess": is_oa,
        "OAStatus": oa_status,
        "ReferencedWorks": get_referenced_works(work)[:1000],
        "RelatedWorks": get_related_works(work)[:1000],
        "Concepts": get_concepts(work)[:500],
        "Topics": get_topics(work)[:500],
        "Domain": tf["Domain"],
        "Field": tf["Field"],
        "Subfield": tf["Subfield"],
        "Institutions": institutions[:250],
        "Languages": get_languages(work),
        "WorkType": work.get("type", "N/A"),
        "DOI": work.get("doi", "N/A"),
        "LandingPage": (work.get("primary_location") or {}).get("landing_page_url", "N/A"),
        "PDFURL": (work.get("primary_location") or {}).get("pdf_url", "N/A")
    }

# ============================================================
# PHASE 1: Skip first SKIP_FIRST results by advancing cursor
# ============================================================
print(f"=== PHASE 1: Skipping first {SKIP_FIRST} results ===")
cursor = "*"
skipped_count = 0
total_fetched = 0

while skipped_count < SKIP_FIRST:
    try:
        data = fetch_openalex_works(QUERY, cursor=cursor)
        results = data.get("results", [])
        if not results:
            print(f"No more results reached after skipping {skipped_count} records.")
            break

        total_fetched += len(results)
        skipped_count += len(results)
        cursor = data.get("meta", {}).get("next_cursor")

        print(f"Skipped so far: {skipped_count}/{SKIP_FIRST} (cursor: {cursor[:20] if cursor else 'None'}...)")

        if not cursor:
            print("No more cursor available - reached end of results")
            break

        # Respect rate limit while skipping
        time.sleep(BASE_DELAY)

    except Exception as e:
        print(f"Error while skipping: {e}")
        time.sleep(BASE_DELAY * 2)

print(f"=== Phase 1 complete: Skipped {skipped_count} records ===")
print(f"Starting collection from cursor: {cursor}")

# ============================================================
# PHASE 2: Collect MAX_RESULTS starting from positioned cursor
# ============================================================
print(f"\n=== PHASE 2: Collecting {MAX_RESULTS} results ===")

records = []
seen_ids = set()
idx = 0
total_found = None

while idx < MAX_RESULTS:
    try:
        data = fetch_openalex_works(QUERY, cursor=cursor)
        if total_found is None:
            total_found = data.get("meta", {}).get("count", 0)

        results = data.get("results", [])
        if not results:
            print("No more results.")
            break

        for work in results:
            if idx >= MAX_RESULTS:
                break

            work_id = work.get("id")
            if work_id and work_id in seen_ids:
                continue
            if work_id:
                seen_ids.add(work_id)

            idx += 1
            record = normalize_record(idx, work)
            records.append(record)

            print(f"Collected: {idx}/{MAX_RESULTS} - {record['Title'][:80]}...")

            if len(records) % SAVE_EVERY == 0:
                pd.DataFrame(records).to_csv(OUT_CSV, index=False)
                print(f"Saved checkpoint: {len(records)} rows")

            sleep_time = BASE_DELAY + random.uniform(-JITTER, JITTER)
            time.sleep(max(0, sleep_time))

        cursor = data.get("meta", {}).get("next_cursor")
        if not cursor:
            break

    except Exception as e:
        print(f"Skipped result due to error: {e}")
        sleep_time = BASE_DELAY + random.uniform(-JITTER, JITTER)
        time.sleep(max(0, sleep_time))

df_WScraping = pd.DataFrame(records)
df_WScraping.to_csv(OUT_CSV, index=False)

print(df_WScraping.to_string(index=False))
print(f"Saved CSV: {OUT_CSV}")
print(f"Total records collected: {len(df_WScraping)}")
print(f"Total records available for query: {total_found}")
print(f"Records skipped: {skipped_count}")

summary = [
    "# OpenAlex Publication Summary",
    "",
    f"- Query: {QUERY}",
    f"- Records skipped: {skipped_count}",
    f"- Total collected: {len(df_WScraping)}",
    f"- Total available: {total_found}",
    f"- Unique works: {df_WScraping['Title'].nunique() if not df_WScraping.empty else 0}",
    f"- Average citations: {round(df['Citations'].mean(), 2) if not df_WScraping.empty else 0}",
    f"- Max citations: {df_WScraping['Citations'].max() if not df_WScraping.empty else 0}",
    "",
    "## Sample rows",
    "",
    df_WScraping.head(15).to_markdown(index=False) if not df_WScraping.empty else "No rows collected."
]

OUT_MD.write_text("\n".join(summary), encoding="utf-8")
print(f"Saved markdown summary: {OUT_MD}")

=== PHASE 1: Skipping first 20000 results ===
Skipped so far: 200/20000 (cursor: IlsxMDkwLjk5MjcsIDE1...)
Skipped so far: 400/20000 (cursor: Ils4MzMuMDQwODMsIDU5...)
Skipped so far: 600/20000 (cursor: Ils3MTAuMTIwMSwgMTIz...)
Skipped so far: 800/20000 (cursor: Ils2MzcuNzU3NzUsIDEz...)
Skipped so far: 1000/20000 (cursor: Ils1NzIuMTYyNywgMTEz...)
Skipped so far: 1200/20000 (cursor: Ils1MzEuMDUxMiwgMTI2...)
Skipped so far: 1400/20000 (cursor: Ils0OTYuMDcwNywgMTU0...)
Skipped so far: 1600/20000 (cursor: Ils0NjUuNjYwMzcsIDEy...)
Skipped so far: 1800/20000 (cursor: Ils0NDAuMjM4MjUsIDE1...)
Skipped so far: 2000/20000 (cursor: Ils0MTcuNzg2MywgODUy...)
Skipped so far: 2200/20000 (cursor: IlszOTguMjI1MjIsIDc3...)
Skipped so far: 2400/20000 (cursor: IlszODIuMDg2NSwgNjk5...)
Skipped so far: 2600/20000 (cursor: IlszNjUuNDE0NDYsIDEx...)
Skipped so far: 2800/20000 (cursor: IlszNTEuMzYzNCwgMTQw...)
Skipped so far: 3000/20000 (cursor: IlszMzYuNzU0OTcsIDcy...)
Skipped so far: 3200/20000 (cursor: IlszMjQ

In [23]:
df_WScraping.head()

NameError: name 'df_WScraping' is not defined

# EDA (Exploratory Data Analysis)

In [8]:
import pandas as pd
import os
print(os.getcwd())

/Users/huss/data-processing/Github/Feminist-Literature-Engine-ML-Web-Scraping-App-/2_notebooks


In [35]:
df_10k = pd.read_csv("/Users/huss/data-processing/Github/Feminist-Literature-Engine-ML-Web-Scraping-App-/1_sources/API_Journal Articles_10k_Titles_Authors_Year_Topic_Publisher.csv", header=0, index_col=0)
df_10k
df_20k = pd.read_csv("/Users/huss/data-processing/Github/Feminist-Literature-Engine-ML-Web-Scraping-App-/1_sources/API_Journal Articles_20k_Titles_Authors_Year_Topic_Publisher.csv", header=0, index_col=0)
df_20k
df_60k = pd.read_csv("/Users/huss/data-processing/Github/Feminist-Literature-Engine-ML-Web-Scraping-App-/1_sources/API_60k_openalex_publications.csv", header=0, index_col=0)
df = pd.concat([df_10k, df_20k, df_60k], axis=0)
df


,Title,Year,Citations,Authors,AuthorOrder,Venue,AbstractInvertedIndex,AbstractText,Publisher,OpenAccess,...,Topics,Domain,Field,Subfield,Institutions,Languages,WorkType,DOI,LandingPage,PDFURL
ID,,,,,,,,,,,,,,,,,,,,,
1,Gender Trouble: Feminism and the Subversion of...,1991.0,28049,Mary McIntosh; Judith Butler,1. Mary McIntosh; 2. Judith Butler,Feminist Review,"{""Preface"": [0, 2], ""(1999)"": [1], ""(1990)"": [...",Preface (1999) Preface (1990) 1. Subjects of S...,SAGE Publishing,False,...,Latin American and Latino Studies; Anarchism a...,Social Sciences,Social Sciences,Cultural Studies,NaN,en,article,https://doi.org/10.2307/1395391,https://doi.org/10.2307/1395391,NaN
2,Situated Knowledges: The Science Question in F...,1988.0,17241,Donna Haraway,1. Donna Haraway,Feminist Studies,NaN,NaN,Feminist Studies,False,...,Feminist Epistemology and Gender Studies; Cont...,Social Sciences,Social Sciences,Sociology and Political Science,NaN,en,article,https://doi.org/10.2307/3178066,https://doi.org/10.2307/3178066,NaN
3,Gender trouble: feminism and the subversion of...,1990.0,7755,NaN,NaN,Choice Reviews Online,NaN,NaN,Association of College and Research Libraries,False,...,Gender Politics and Representation; Historical...,Social Sciences,Social Sciences,Gender Studies,NaN,en,article,https://doi.org/10.5860/choice.28-1264,https://doi.org/10.5860/choice.28-1264,NaN
4,Situated Knowledges: The Science Question in F...,1988.0,7010,Donna Haraway,1. Donna Haraway,PhilPapers (PhilPapers Foundation),"{""Academic"": [0], ""and"": [1, 23, 34, 44, 60, 6...",Academic and activist feminist inquiry has rep...,NaN,True,...,Interdisciplinary Research and Collaboration; ...,Social Sciences,Decision Sciences,Information Systems and Management,NaN,en,article,NaN,https://philarchive.org/rec/HARSKT,https://philpapers.org/archive/HARSKT.pdf
5,The science question in feminism,1987.0,3829,Kristin Waters,1. Kristin Waters,Women s Studies International Forum,NaN,NaN,Elsevier BV,False,...,Species Distribution and Climate Change; Susta...,Physical Sciences,Environmental Science,Ecological Modeling,NaN,en,article,https://doi.org/10.1016/0277-5395(87)90077-x,https://doi.org/10.1016/0277-5395(87)90077-x,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59996,To tell the right story,2013.0,13,Erik Eriksson,1. Erik Eriksson,Journal of Comparative Social Work,"{""From"": [0], ""the"": [1, 46, 68, 85, 89, 93, 1...",From the starting point of narrative ethnograp...,University of Stavanger,True,...,Mental Health and Patient Involvement; Mental ...,Health Sciences,Health Professions,General Health Professions,Lund University,en,article,https://doi.org/10.31265/jcsw.v8i2.103,https://doi.org/10.31265/jcsw.v8i2.103,https://journals.uis.no/index.php/JCSW/article...
59997,Generational Difference in Feminist Identities...,2009.0,0,Catherine E. Harnois,1. Catherine E. Harnois,DOAJ (DOAJ: Directory of Open Access Journals),"{""Studies"": [0], ""of"": [1, 34, 65, 84], ""the"":...",Studies of the general population have found s...,NaN,False,...,Youth Education and Societal Dynamics,Social Sciences,Social Sciences,Sociology and Political Science,NaN,en,article,NaN,https://doaj.org/article/c300e1880fe34eb6baf2d...,NaN
59998,Visionary Pragmatism and the Value of Privacy ...,2010.0,9,Danielle Keats Citron; Leslie Meltzer Henry,1. Danielle Keats Citron; 2. Leslie Meltzer Henry,Michigan Law Review,"{""Part"": [0, 83, 120], ""I"": [1], ""of"": [2, 9, ...",Part I of our Review discusses the central pre...,University of Michigan Law School,True,...,"Law, Rights, and Freedoms; Legal Systems and J...",Social Sciences,Social Sciences,Sociology and Political Science,NaN,en,article,https://doi.org/10.36644/mlr.108.6.visionary,https://doi.org/10.36644/mlr.108.6.visionary,NaN


In [ ]:

#-----------------------------------------WebScraping-----------------------------------------
df_ws = pd.read_csv("/Users/huss/data-processing/Github/Feminist-Literature-Engine-ML-Web-Scraping-App-/1_sources/WebScraping_1000_feminist_books_goodreads.csv", header=0, index_col=0)
df_ws


#-----------------------------------------WebScraping-----------------------------------------
#webscraping data cannot be used for the task because for two reasons 1 the only common column with the 80k data is the title 2 there are many fiction or non-scientific books which got filtered in

In [31]:
df.columns = (
    df.columns
    .str.lower()
    .str.replace('(', '_', regex=False)   # replace ( with _
    .str.replace('/', '_', regex=False)   # replace ( with _
    .str.replace(')', '', regex=False)    # remove )
    .str.replace(r'[^\w\s]', '', regex=True)  # remove other special chars
    .str.replace(' ', '_')                # spaces → underscore
    .str.replace('__', '_')
)
df.columns



Index(['title', 'year', 'citations', 'authors', 'authororder', 'venue',
       'abstractinvertedindex', 'abstracttext', 'publisher', 'openaccess',
       'oastatus', 'referencedworks', 'relatedworks', 'concepts', 'topics',
       'domain', 'field', 'subfield', 'institutions', 'languages', 'worktype',
       'doi', 'landingpage', 'pdfurl'],
      dtype='str')

In [10]:
import numpy as np
#Explore all data
#print("\n", "Head:","\n", df.head(5))
#print("\n", "Tail:","\n", df.tail(5))
print("\n", "Null Values:","\n", df.isna().sum())




 Null Values: 
 title                       44
year                       109
citations                    0
authors                   3500
authororder               3500
venue                    12883
abstractinvertedindex    19808
abstracttext             19865
publisher                27358
openaccess                   0
oastatus                     0
referencedworks          43319
relatedworks              7783
concepts                   368
topics                     688
domain                     688
field                      688
subfield                   688
institutions             44620
languages                 1142
worktype                     0
doi                      17754
landingpage               1735
pdfurl                   58531
dtype: int64


In [11]:
df = df.drop(["pdfurl", "venue", "abstractinvertedindex", "abstracttext", "publisher", "referencedworks", "institutions", "doi", "landingpage"], axis=1)
df

,title,year,citations,authors,authororder,openaccess,oastatus,relatedworks,concepts,topics,domain,field,subfield,languages,worktype
ID,,,,,,,,,,,,,,,
1,Gender Trouble: Feminism and the Subversion of...,1991.0,28049,Mary McIntosh; Judith Butler,1. Mary McIntosh; 2. Judith Butler,False,closed,https://openalex.org/W2982445252; https://open...,Subversion; Feminism; Gender studies; Sociolog...,Latin American and Latino Studies; Anarchism a...,Social Sciences,Social Sciences,Cultural Studies,en,article
2,Situated Knowledges: The Science Question in F...,1988.0,17241,Donna Haraway,1. Donna Haraway,False,closed,https://openalex.org/W1946080426; https://open...,Situated; Perspective (graphical); Privilege (...,Feminist Epistemology and Gender Studies; Cont...,Social Sciences,Social Sciences,Sociology and Political Science,en,article
3,Gender trouble: feminism and the subversion of...,1990.0,7755,NaN,NaN,False,closed,https://openalex.org/W2038600245; https://open...,Subversion; Feminism; Identity (music); Gender...,Gender Politics and Representation; Historical...,Social Sciences,Social Sciences,Gender Studies,en,article
4,Situated Knowledges: The Science Question in F...,1988.0,7010,Donna Haraway,1. Donna Haraway,True,green,https://openalex.org/W2753533763; https://open...,Situated; Epistemology; Sociology; Relativism;...,Interdisciplinary Research and Collaboration; ...,Social Sciences,Decision Sciences,Information Systems and Management,en,article
5,The science question in feminism,1987.0,3829,Kristin Waters,1. Kristin Waters,False,closed,https://openalex.org/W3114154697; https://open...,Bridge (graph theory); Sustainability; Knowled...,Species Distribution and Climate Change; Susta...,Physical Sciences,Environmental Science,Ecological Modeling,en,article
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59996,To tell the right story,2013.0,13,Erik Eriksson,1. Erik Eriksson,True,diamond,https://openalex.org/W2393609567; https://open...,Narrative; Personal narrative; Ethnography; Ch...,Mental Health and Patient Involvement; Mental ...,Health Sciences,Health Professions,General Health Professions,en,article
59997,Generational Difference in Feminist Identities...,2009.0,0,Catherine E. Harnois,1. Catherine E. Harnois,False,closed,https://openalex.org/W2122261870; https://open...,Gender studies; Sociology,Youth Education and Societal Dynamics,Social Sciences,Social Sciences,Sociology and Political Science,en,article
59998,Visionary Pragmatism and the Value of Privacy ...,2010.0,9,Danielle Keats Citron; Leslie Meltzer Henry,1. Danielle Keats Citron; 2. Leslie Meltzer Henry,True,gold,https://openalex.org/W266204379; https://opena...,Pragmatism; Skepticism; Privacy policy; Autono...,"Law, Rights, and Freedoms; Legal Systems and J...",Social Sciences,Social Sciences,Sociology and Political Science,en,article


In [12]:
print("\n", "Null Values:","\n", df.isna().sum())



 Null Values: 
 title             44
year             109
citations          0
authors         3500
authororder     3500
openaccess         0
oastatus           0
relatedworks    7783
concepts         368
topics           688
domain           688
field            688
subfield         688
languages       1142
worktype           0
dtype: int64


In [13]:
df = df.dropna(how="any")
df
print("\n", "Null Values:","\n", df.isna().sum())


 Null Values: 
 title           0
year            0
citations       0
authors         0
authororder     0
openaccess      0
oastatus        0
relatedworks    0
concepts        0
topics          0
domain          0
field           0
subfield        0
languages       0
worktype        0
dtype: int64


In [14]:
print("\n", "Info:","\n", df.info())
#print("\n", "Info Memory Usage:","\n", df.info(memory_usage="deep"))


<class 'pandas.DataFrame'>
Index: 68692 entries, 1 to 60000
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         68692 non-null  str    
 1   year          68692 non-null  float64
 2   citations     68692 non-null  int64  
 3   authors       68692 non-null  str    
 4   authororder   68692 non-null  str    
 5   openaccess    68692 non-null  bool   
 6   oastatus      68692 non-null  str    
 7   relatedworks  68692 non-null  str    
 8   concepts      68692 non-null  str    
 9   topics        68692 non-null  str    
 10  domain        68692 non-null  str    
 11  field         68692 non-null  str    
 12  subfield      68692 non-null  str    
 13  languages     68692 non-null  str    
 14  worktype      68692 non-null  str    
dtypes: bool(1), float64(1), int64(1), str(12)
memory usage: 63.2 MB

 Info: 
 None


In [15]:
print("\n", "Describe:","\n", df.describe())
print("\n", "Describe All:","\n", df.describe(include="all"))
print("\n", "Shape:","\n", df.shape)
print("\n", "DTypes:","\n", df.dtypes)
print("\n", "Columns:","\n", df.columns)
#print("\n", "Index:","\n", df.index)
print("\n", "Values:","\n", df.values)
print("\n", "Empty:","\n", df.empty)
#print("\n", "NDim:","\n", df.ndim)
print("\n", "Size:","\n", df.size)
#print("\n", "Memory True:","\n", df.memory_usage(deep=True))
#print("\n", "Transpose:","\n", df.T)


 Describe: 
                year     citations
count  68692.000000  68692.000000
mean    2011.384397     29.556135
std       11.038538    227.479699
min     1814.000000      0.000000
25%     2005.000000      1.000000
50%     2015.000000      4.000000
75%     2020.000000     22.000000
max     2026.000000  28049.000000

 Describe All: 
                title          year     citations      authors     authororder  \
count          68692  68692.000000  68692.000000        68692           68692   
unique         66081           NaN           NaN        51412           51413   
top     Introduction           NaN           NaN  Karen Offen  1. Karen Offen   
freq             155           NaN           NaN           39              39   
mean             NaN   2011.384397     29.556135          NaN             NaN   
std              NaN     11.038538    227.479699          NaN             NaN   
min              NaN   1814.000000      0.000000          NaN             NaN   
25%           

In [16]:
#split cols into different columns 
numerical_cols = df.select_dtypes(include=np.number)
print(numerical_cols)

#be careful because some categorical values my sip into numerical_cols (numbers that represent category)
categorical_cols = df.select_dtypes(exclude=np.number)
print(categorical_cols)

         year  citations
ID                      
1      1991.0      28049
2      1988.0      17241
4      1988.0       7010
5      1987.0       3829
6      1995.0       5048
...       ...        ...
59996  2013.0         13
59997  2009.0          0
59998  2010.0          9
59999  2019.0          9
60000  2009.0         10

[68692 rows x 2 columns]
                                                   title  \
ID                                                         
1      Gender Trouble: Feminism and the Subversion of...   
2      Situated Knowledges: The Science Question in F...   
4      Situated Knowledges: The Science Question in F...   
5                       The science question in feminism   
6      Unbearable Weight: Feminism, Western Culture a...   
...                                                  ...   
59996                            To tell the right story   
59997  Generational Difference in Feminist Identities...   
59998  Visionary Pragmatism and the Value of Priv

In [17]:
df["year"] = df["year"].astype(int)


In [18]:
df.head()

,title,year,citations,authors,authororder,openaccess,oastatus,relatedworks,concepts,topics,domain,field,subfield,languages,worktype
ID,,,,,,,,,,,,,,,
1,Gender Trouble: Feminism and the Subversion of...,1991,28049,Mary McIntosh; Judith Butler,1. Mary McIntosh; 2. Judith Butler,False,closed,https://openalex.org/W2982445252; https://open...,Subversion; Feminism; Gender studies; Sociolog...,Latin American and Latino Studies; Anarchism a...,Social Sciences,Social Sciences,Cultural Studies,en,article
2,Situated Knowledges: The Science Question in F...,1988,17241,Donna Haraway,1. Donna Haraway,False,closed,https://openalex.org/W1946080426; https://open...,Situated; Perspective (graphical); Privilege (...,Feminist Epistemology and Gender Studies; Cont...,Social Sciences,Social Sciences,Sociology and Political Science,en,article
4,Situated Knowledges: The Science Question in F...,1988,7010,Donna Haraway,1. Donna Haraway,True,green,https://openalex.org/W2753533763; https://open...,Situated; Epistemology; Sociology; Relativism;...,Interdisciplinary Research and Collaboration; ...,Social Sciences,Decision Sciences,Information Systems and Management,en,article
5,The science question in feminism,1987,3829,Kristin Waters,1. Kristin Waters,False,closed,https://openalex.org/W3114154697; https://open...,Bridge (graph theory); Sustainability; Knowled...,Species Distribution and Climate Change; Susta...,Physical Sciences,Environmental Science,Ecological Modeling,en,article
6,"Unbearable Weight: Feminism, Western Culture a...",1995,5048,Mimi Nichter; Susan Bordo,1. Mimi Nichter; 2. Susan Bordo,False,closed,https://openalex.org/W2748952813; https://open...,Feminism; Gender studies; Sociology; Political...,French Historical and Cultural Studies,Social Sciences,Arts and Humanities,History,en,article


In [20]:
df.to_csv("/Users/huss/data-processing/Github/Feminist-Literature-Engine-ML-Web-Scraping-App-/1_sources/clean_merged_80k.csv", sep="\t", index=False)

In [21]:
import csv
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Goodreads feminist book lists to scrape
LISTS = [
    ("https://www.goodreads.com/list/show/62.Best_Feminist_Books", "Best Feminist Books - 2,752 books"),
    ("https://www.goodreads.com/list/show/46.Best_Feminist_Fiction", "Best Feminist Fiction - 1,503 books"),
    ("https://www.goodreads.com/shelf/show/feminism", "Popular Feminism Shelf"),
    ("https://www.goodreads.com/list/tag?id=feminism&ref=ls_ts", "Feminism Tag List"),
]

OUTPUT_CSV = "1000_feminist_books_goodreads.csv"
TARGET_BOOKS = 1000

def setup_driver():
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36")
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(60)
    return driver

def scrape_page(url, page_num=1):
    """Scrape one page of books"""
    books = []
    
    try:
        driver = setup_driver()
        
        # Handle pagination
        if page_num == 1:
            full_url = url
        else:
            if "show" in url:
                full_url = f"{url}?page={page_num}"
            elif "shelf" in url:
                full_url = f"{url}?page={page_num}"
            else:
                start = (page_num - 1) * 50 + 1
                full_url = f"{url}&start={start}"
        
        print(f"  → Page {page_num}: {full_url}")
        driver.get(full_url)
        time.sleep(3)
        
        # JavaScript extraction
        script = """
        var books = [];
        var rows = document.querySelectorAll('tr[itemtype="http://schema.org/Book"]');
        
        for (var i = 0; i < rows.length; i++) {
            var row = rows[i];
            var book = {};
            
            var titleEl = row.querySelector('.bookTitle');
            if (titleEl) {
                book.title = titleEl.textContent.trim();
                book.book_url = titleEl.href;
            }
            
            var authorEl = row.querySelector('.authorName');
            if (authorEl) {
                book.author = authorEl.textContent.trim();
                book.author_url = authorEl.href;
            }
            
            var spans = row.querySelectorAll('span');
            for (var j = 0; j < spans.length; j++) {
                var text = spans[j].textContent;
                if (text.includes('avg rating')) {
                    var match = text.match(/([\\d.]+)/);
                    if (match) book.avg_rating = parseFloat(match[1]);
                }
                if (text.includes('ratings')) {
                    var match = text.match(/([\\d,]+)/);
                    if (match) book.ratings_count = parseInt(match[1].replace(/,/g, ''));
                }
                if (text.includes('votes')) {
                    var match = text.match(/([\\d,]+)/);
                    if (match) book.votes_count = parseInt(match[1].replace(/,/g, ''));
                }
                if (text.includes('published')) {
                    var match = text.match(/published\\s+([\\d]{4})/);
                    if (match) book.published_year = match[1];
                }
                if (text.includes('reviews')) {
                    var match = text.match(/([\\d,]+)/);
                    if (match) book.reviews_count = parseInt(match[1].replace(/,/g, ''));
                }
            }
            
            var coverImg = row.querySelector('img.bookCover');
            if (coverImg) book.cover_url = coverImg.src;
            
            if (book.title) books.push(book);
        }
        return books;
        """
        
        books = driver.execute_script(script)
        
    except Exception as e:
        print(f"Error: {e}")
    finally:
        driver.quit()
    
    return books

def scrape_all_lists(target=1000):
    """Scrape all lists with pagination"""
    all_books = []
    seen_urls = set()
    
    print("=" * 80)
    print("📚 GOODREADS FEMINIST BOOKS SCRAPER - 1000 BOOKS")
    print("=" * 80)
    print(f"Target: {target} books")
    print()
    
    for list_url, list_name in LISTS:
        print(f"\n📖 {list_name}")
        print(f"URL: {list_url}")
        
        page = 1
        max_pages = 20
        
        while len(all_books) < target and page <= max_pages:
            books = scrape_page(list_url, page)
            
            # Filter duplicates
            new_books = []
            for book in books:
                url = book.get('book_url', '')
                if url and url not in seen_urls:
                    seen_urls.add(url)
                    book['source'] = list_name.split(' - ')[0]
                    new_books.append(book)
            
            all_books.extend(new_books)
            print(f"   ✓ +{len(new_books)} books (Total: {len(all_books)}/{target})")
            
            if len(new_books) == 0 or len(all_books) >= target:
                break
            
            page += 1
            time.sleep(1)
        
        if len(all_books) >= target:
            print(f"\n🎯 Reached target of {target} books!")
            break
    
    all_books = all_books[:target]
    
    print("\n" + "=" * 80)
    print(f"✅ SUCCESS! Scraped {len(all_books)} feminist books")
    print("=" * 80)
    
    return all_books

def save_csv(books, filename):
    if not books:
        print("❌ No books to save")
        return
    
    fieldnames = list(books[0].keys())
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(books)
    
    print(f"💾 Saved {len(books)} books to {filename}")
    print(f"   Columns: {len(fieldnames)} - {fieldnames}")

def print_stats(books):
    if not books:
        return
    
    df = pd.DataFrame(books)
    
    print("\n" + "=" * 80)
    print("📊 FEMINIST BOOKS STATISTICS")
    print("=" * 80)
    
    print(f"\n📚 TOTAL BOOKS: {len(df)}")
    
    # Data completeness
    print(f"\n📈 DATA COMPLETENESS:")
    for col in df.columns:
        complete = (df[col].notna() & (df[col] != '')).sum()
        pct = complete / len(df) * 100
        print(f"   {col}: {pct:.1f}%")
    
    # Ratings
    if 'avg_rating' in df.columns and df['avg_rating'].notna().any():
        print(f"\n⭐ RATING STATISTICS:")
        print(f"   Average: {df['avg_rating'].mean():.2f}")
        print(f"   Highest: {df['avg_rating'].max():.2f}")
        print(f"   Median: {df['avg_rating'].median():.2f}")
    
    # Ratings count
    if 'ratings_count' in df.columns and df['ratings_count'].notna().any():
        print(f"\n📊 RATINGS:")
        print(f"   Total: {df['ratings_count'].sum():,}")
        print(f"   Average: {df['ratings_count'].mean():,.0f}")
        print(f"   Max: {df['ratings_count'].max():,}")
    
    # Top 30 by ratings
    print(f"\n🏆 TOP 30 MOST RATED FEMINIST BOOKS:")
    if 'ratings_count' in df.columns:
        top = df.nlargest(30, 'ratings_count')
        for idx, row in top.iterrows():
            rating = row.get('avg_rating', 'N/A')
            votes = row.get('votes_count', 0)
            print(f"   {idx+1}. {row['title']} by {row['author']}")
            print(f"      ⭐ {rating} | 📊 {row['ratings_count']:,} ratings | 🏆 {votes:,} votes")
    
    # Top 20 by rating (min 500 ratings)
    print(f"\n🌟 TOP 20 HIGHEST RATED (min 500 ratings):")
    if 'avg_rating' in df.columns and 'ratings_count' in df.columns:
        qualified = df[(df['avg_rating'].notna()) & (df['ratings_count'].notna()) & (df['ratings_count'] >= 500)]
        if len(qualified) > 0:
            top = qualified.nlargest(20, 'avg_rating')
            for idx, row in top.iterrows():
                print(f"   {idx+1}. {row['title']} by {row['author']}")
                print(f"      ⭐ {row['avg_rating']:.2f} | 📊 {row['ratings_count']:,} ratings")
    
    # Highest voted on lists
    print(f"\n🔥 TOP 20 MOST POPULAR ON LIST (by votes):")
    if 'votes_count' in df.columns:
        top = df.nlargest(20, 'votes_count')
        for idx, row in top.iterrows():
            ratings = row.get('ratings_count', 0)
            print(f"   {idx+1}. {row['title']} by {row['author']} - {row['votes_count']:,} votes ({ratings:,} ratings)")
    
    # Publication years
    if 'published_year' in df.columns and df['published_year'].notna().any():
        print(f"\n📅 PUBLICATION YEAR DISTRIBUTION:")
        df_clean = df[df['published_year'].notna()]
        df_clean['year'] = pd.to_numeric(df_clean['published_year'], errors='coerce')
        by_decade = df_clean.groupby(df_clean['year'] // 10 * 10).size()
        for decade, count in by_decade.sort_index().items():
            pct = count / len(df_clean) * 100
            print(f"   {int(decade)}s: {count} books ({pct:.1f}%)")
    
    # Top authors
    print(f"\n👥 TOP 30 AUTHORS (most books on lists):")
    if 'author' in df.columns:
        authors = df['author'].value_counts().head(30)
        for author, count in authors.items():
            print(f"   {author}: {count} books")
    
    # Sources
    if 'source' in df.columns:
        print(f"\n📚 SOURCE LIST BREAKDOWN:")
        sources = df['source'].value_counts()
        for source, count in sources.items():
            print(f"   {source}: {count} books")

def main():
    print("\n🚀 Starting scraper for 1000 feminist books...\n")
    
    books = scrape_all_lists(TARGET_BOOKS)
    
    if books:
        save_csv(books, OUTPUT_CSV)
        print_stats(books)
        
        # Save top books
        df = pd.DataFrame(books)
        if 'ratings_count' in df.columns:
            top30 = df.nlargest(30, 'ratings_count')
            top30.to_csv("top_30_feminist_books_by_ratings.csv", index=False)
            print(f"\n✅ Top 30 saved to: top_30_feminist_books_by_ratings.csv")
        
        print("\n" + "=" * 80)
        print("✅ ALL DONE!")
        print(f"   Main file: {OUTPUT_CSV} ({len(books)} books)")
        print(f"   Top 30: top_30_feminist_books_by_ratings.csv")
        print("=" * 80 + "\n")
    
    return books

if __name__ == "__main__":
    books = main()



🚀 Starting scraper for 1000 feminist books...

📚 GOODREADS FEMINIST BOOKS SCRAPER - 1000 BOOKS
Target: 1000 books


📖 Best Feminist Books - 2,752 books
URL: https://www.goodreads.com/list/show/62.Best_Feminist_Books
  → Page 1: https://www.goodreads.com/list/show/62.Best_Feminist_Books
   ✓ +100 books (Total: 100/1000)
  → Page 2: https://www.goodreads.com/list/show/62.Best_Feminist_Books?page=2
   ✓ +100 books (Total: 200/1000)
  → Page 3: https://www.goodreads.com/list/show/62.Best_Feminist_Books?page=3
